# Data Wrangling #

In [17]:
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3
import re

# Connect to the database
conn = sqlite3.connect("./data/nutrition.db")
cur = conn.cursor()

In [2]:
food = pd.read_sql_query("SELECT * FROM food", conn)
food_nutrient = pd.read_sql_query("SELECT * FROM food_nutrient", conn)
nutrient = pd.read_sql_query("SELECT * FROM nutrient", conn)
wal_price = pd.read_sql_query("SELECT * FROM walmart_price", conn)
wf_price = pd.read_sql_query("SELECT * FROM wholefoods_price", conn)

In [3]:
def normalize_text(str):
    if pd.isna(str):
        return ""
    str = str.lower()
    str = re.sub(r'[^a-z0-9\s]', ' ', str)
    str = re.sub(r'\s+', ' ', str).strip()
    return str

food["clean_desc"] = food["description"].apply(normalize_text)
wal_price["clean_name"] = wal_price["product_name"].apply(normalize_text)
wf_price["clean_name"] = wf_price["product_name"].apply(normalize_text)

food["clean_brand_owner"] = food["brand_owner"].apply(normalize_text)
food["clean_brand"] = food["brand_name"].apply(normalize_text)
food["clean_subbrand"] = food["subbrand_name"].apply(normalize_text)

wal_price["clean_brand"] = wal_price["brand"].apply(normalize_text)
wf_price["clean_brand"] = wf_price["brand"].apply(normalize_text)

In [4]:
food_brand_owners = food["clean_brand_owner"].dropna().unique().tolist()
food_brands = food["clean_brand"].dropna().unique().tolist()
food_subbrands = food["clean_subbrand"].dropna().unique().tolist()

wal_brands = wal_price["clean_brand"].dropna().unique().tolist()
wf_brands = wf_price["clean_brand"].dropna().unique().tolist()

In [25]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pathlib

model: SentenceTransformer = SentenceTransformer('all-MiniLM-L6-v2')

def load_embedding(path: pathlib.Path):
    if path.exists():
        return np.load(path)
    else:
        return None

def save_embedding(embedding, path: pathlib.Path):
    np.save(path, embedding)

wf_brand_emb = load_embedding(pathlib.Path('./data/wf_brand_emb.npy'))
if wf_brand_emb is None:
    print("Embedding could not be loaded")
    wf_brand_emb = model.encode(wf_brands, show_progress_bar=True)
    save_embedding(wf_brand_emb, pathlib.Path('./data/wf_brand_emb'))

wal_brand_emb = load_embedding(pathlib.Path('./data/wal_brand_emb.npy'))
if wal_brand_emb is None:
    print("Embedding could not be loaded")
    wal_brand_emb = model.encode(wal_brands, show_progress_bar=True)
    save_embedding(wal_brand_emb, pathlib.Path('./data/wal_brand_emb'))

food_brand_owner_emb = load_embedding(pathlib.Path('./data/food_brand_owner_emb.npy'))
if food_brand_owner_emb is None:
    print("Embedding could not be loaded")
    food_brand_owner_emb = model.encode(food_brand_owners, show_progress_bar=True)
    save_embedding(food_brand_owner_emb, pathlib.Path('./data/food_brand_owner_emb'))

food_brand_emb = load_embedding(pathlib.Path('./data/food_brand_emb.npy'))
if food_brand_emb is None:
    print("Embedding could not be loaded")
    food_brand_emb = model.encode(food_brands, show_progress_bar=True)
    save_embedding(food_brand_emb, pathlib.Path('./data/food_brand_emb'))

food_subbrand_emb = load_embedding(pathlib.Path('./data/food_subbrand_emb.npy'))
if food_subbrand_emb is None:
    print("Embedding could not be loaded")
    food_subbrand_emb = model.encode(food_subbrands, show_progress_bar=True)
    save_embedding(food_subbrand_emb, pathlib.Path('./data/food_subbrand_emb'))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2107.62it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
from sentence_transformers import util
import numpy as np

SIMILARITY_THRESH = 0.95

wf_sim_matrix = util.cos_sim(wf_brand_emb, food_brand_emb)
wf_s_sim_matrix = util.cos_sim(wf_brand_emb, food_subbrand_emb)
wal_sim_matrix = util.cos_sim(wal_brand_emb, food_brand_emb)
wal_s_sim_matrix = util.cos_sim(wal_brand_emb, food_subbrand_emb)

wf_to_food = {}
for i, wf_brand in enumerate(wf_brands):
    best_idx = np.argmax(wf_sim_matrix[i]).item()
    best_match = food_brands[best_idx]
    best_score = wf_sim_matrix[i][best_idx].item()

    if best_score > SIMILARITY_THRESH:
        wf_to_food[wf_brand] = (best_match, best_score)
    else:
        sub_best_idx = np.argmax(wf_s_sim_matrix[i]).item()
        sub_best_match = food_brands[sub_best_idx]
        sub_best_score = wf_s_sim_matrix[i][sub_best_idx].item()

        if sub_best_score > best_score:
            wf_to_food[wf_brand] = (sub_best_match, sub_best_score)
        else:
            wf_to_food[wf_brand] = (best_match, best_score)

wal_to_food = {}
for i, wal_brand in enumerate(wal_brands):
    best_idx = np.argmax(wal_sim_matrix[i]).item()
    best_match = food_brands[best_idx]
    best_score = wal_sim_matrix[i][best_idx].item()
    
    if best_score > SIMILARITY_THRESH:
        wal_to_food[wal_brand] = (best_match, best_score)
    else:
        sub_best_idx = np.argmax(wal_s_sim_matrix[i]).item()
        sub_best_match = food_brands[sub_best_idx]
        sub_best_score = wal_s_sim_matrix[i][sub_best_idx].item()

        if sub_best_score > best_score:
            wal_to_food[wal_brand] = (sub_best_match, sub_best_score)
        else:
            wal_to_food[wal_brand] = (best_match, best_score)

wf_price['udsa_brand_match'] = wf_price['clean_brand'].map(lambda b: wf_to_food.get(b, (None, 0))[0])
wf_price['udsa_brand_score'] = wf_price['clean_brand'].map(lambda b: wf_to_food.get(b, (None, 0))[1])
wal_price['udsa_brand_match'] = wal_price['clean_brand'].map(lambda b: wal_to_food.get(b, (None, 0))[0])
wal_price['udsa_brand_score'] = wal_price['clean_brand'].map(lambda b: wal_to_food.get(b, (None, 0))[1])

In [ ]:
food_desc_emb_path = pathlib.Path('./data/food-desc-emb')
food_desc_emb = load_embedding(food_desc_emb_path)
if food_desc_emb is None:
    food_desc_emb = model.encode(food.clean_desc, show_progress_bar=True)
    save_embedding(food_desc_emb, food_desc_emb_path)

Batches:   0%|          | 4/19283 [00:01<2:09:42,  2.48it/s]


KeyboardInterrupt: 

In [ ]:
for brand in wf_brands:
    branded_items = wf_price[wf_price.brand == brand]
    matched_brands = food[food.subbrand_name == wf_to_food[brand][0] | food.brand_name == wf_to_food[brand][0]]

    wf_item_emb = model.encode(branded_items.clean_name)
    item_sim_matrix = util.cos_sim(wf_item_emb, food_desc_emb)

    # Use sim matrix to match w/ fdc id and add it as a feature
    # Maybe this should be its own DF/table in the db?
    

UndefinedVariableError: name 'wf_to_food' is not defined

In [ ]:
from rapidfuzz import process



In [ ]:
food_names = food["clean_desc"].tolist()

def get_candidates(query, k=5):
    get_candidates.calls += 1
    if get_candidates.calls < 10 or (get_candidates.calls < 100 and get_candidates.calls % 10 == 0) or (get_candidates.calls < 1000 and get_candidates.calls % 100 == 0) or (get_candidates.calls < 10000 and get_candidates.calls % 1000 == 0) or get_candidates.calls % 10000 == 0:
        print(get_candidates.calls)

    matches = process.extract(query, food_names, limit=k)
    return [(m[0], m[1]) for m in matches]

get_candidates.calls = 0

wf_price["candidates"] = wf_price["clean_name"].apply(get_candidates)

1
2
3
4
5
6
7
8
9
10


KeyboardInterrupt: 

In [ ]:
def compute_score(store_row, food_row, text_score):
    text_sim = text_score / 100

    brand_sim = 1 if store_row["clean_brand"] == food_row["clean_brand_owner"] or store_row["clean_brand"] == food_row["clean_brand"] or store_row["clean_brand"] == food_row["clean_subbrand"] else 0

    return 0.7 * text_sim + 0.3 * brand_sim

In [ ]:
def match_product(row, score_thresh):
    best_score = score_thresh
    best_fdc = None

    for cand_name, text_score in row["candidates"]:
        food_row = food[food["clean_desc"] == cand_name].iloc[0]

        score = compute_score(row, food_row, text_score)

        if score > best_score:
            best_score = score
            best_fdc = food_row["fdc_id"]

    return best_fdc, best_score

wal_price[["fdc_id", "match_score"]] = wal_price.apply(
    lambda row: pd.Series(match_product(row, 0)),
    axis=1
)

In [ ]:
conn.close()